# Travaux: Chandra OCR 2 over the UN records (1946-66)
Reads the remaining pages of the queue with the original Chandra OCR 2 (`datalab-to/chandra-ocr-2`) served by vLLM on the
two T4 GPUs, with the same prompt and text conversion as the laptop run (`pipeline/hpc/chandra_client.py`). Settings needed:
**Accelerator: GPU T4 x2**, **Internet: on**, the dataset **travaux-scans** attached. Results: `/kaggle/working/results`
(zipped as `results.zip` at the end), copied back into `data/ocr/` on the laptop. Pages already in `results` of an earlier
version are not read again if that output is attached as input (see the last cell).

In [ ]:
# vLLM built for CUDA 12.9 (Chandra OCR 2 is a Qwen3.5 model, so a recent vLLM is needed); after it, Kaggle's torchaudio and
# torchcodec are removed: they are built for another PyTorch and transformers / vLLM import them on start (audio only)
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv
!pip install -q "https://github.com/vllm-project/vllm/releases/download/v0.31.0/vllm-0.31.0+cu129-cp38-abi3-manylinux_2_28_x86_64.whl" pymupdf openai --extra-index-url https://download.pytorch.org/whl/cu129 2>&1 | grep -v "requires\|incompatible" | tail -3
!pip uninstall -y -q torchaudio torchcodec
!python -c "import torch, vllm; print('torch', torch.__version__, 'cuda', torch.version.cuda, 'gpus', torch.cuda.device_count(), '| vllm', vllm.__version__)"
import glob, os
# the dataset's root is where the queue is (an earlier run's output, attached as input, also holds a copy of the code)
ROOT = os.path.dirname(glob.glob('/kaggle/input/**/chandra_queue_remaining.json', recursive=True)[0])
print('code and scans in', ROOT, '|', len(glob.glob(ROOT + '/raw/*.pdf')), 'PDFs')
# the client as committed after the first run waits 30 min for an answer (the dataset's copy has the OpenAI client's 600 s default,
# which timed out 25 of 1,986 pages): a working copy of the code with that one change
CODE = '/kaggle/working/code'
!rm -rf {CODE} && mkdir -p {CODE} && cp -r {ROOT}/pipeline {ROOT}/eval {CODE}/
!grep -q 'timeout=' {CODE}/pipeline/hpc/chandra_client.py || sed -i 's/api_key="local")/api_key="local", timeout=1800)/' {CODE}/pipeline/hpc/chandra_client.py
!grep -c 'timeout=' {CODE}/pipeline/hpc/chandra_client.py

In [ ]:
# results of earlier runs, if their output was attached as input: copy them in, so those pages are skipped
import shutil, subprocess
os.makedirs('/kaggle/working/results', exist_ok=True)
for z in glob.glob('/kaggle/input/**/results.zip', recursive=True):
    subprocess.run(['unzip', '-q', '-n', z, '-d', '/kaggle/working'], check=False)
print(sum(1 for _ in glob.iglob('/kaggle/working/results/*/chandra/*.txt')), 'pages already read')

In [ ]:
# the model server: fp16 (the T4 has no bf16), split over the two GPUs; eager mode, no custom all-reduce, NCCL P2P off (the
# settings that started on Kaggle's T4s, 17 Oct 2026); its own process group, so an interrupted cell does not kill it. The first
# start compiles Triton kernels for the T4 (ptxas): about 13 minutes with no log lines, which is not a hang.
import os, subprocess, time, urllib.request
env = dict(os.environ, NCCL_P2P_DISABLE='1', NCCL_IB_DISABLE='1')
server = subprocess.Popen(['vllm', 'serve', 'datalab-to/chandra-ocr-2', '--served-model-name', 'chandra',
                           '--dtype', 'half', '--tensor-parallel-size', '2', '--max-model-len', '16384',
                           '--gpu-memory-utilization', '0.92', '--limit-mm-per-prompt', '{"image": 1}',
                           '--enforce-eager', '--disable-custom-all-reduce', '--port', '8000'],
                          stdout=open('/kaggle/working/vllm.log', 'w'), stderr=subprocess.STDOUT, start_new_session=True, env=env)
for i in range(360):
    try:
        urllib.request.urlopen('http://localhost:8000/health', timeout=5); print('server up after', i * 10, 's'); break
    except Exception:
        if server.poll() is not None: print(open('/kaggle/working/vllm.log').read()[-3000:]); raise SystemExit('vLLM stopped')
        time.sleep(10)

In [ ]:
# a test on 4 pages first (prints the time a page), then the run: no new page after 11 hours
!python {CODE}/pipeline/hpc/chandra_client.py --queue {ROOT}/chandra_queue_remaining.json --raw {ROOT}/raw --out /kaggle/working/results --workers 16 --limit 4
!tail -4 /kaggle/working/results/chandra_run.jsonl

In [ ]:
# reading time: 10.5 h for a full 12-h session; less when the week's GPU quota is short (startup ~25 min, pages in flight
# finish up to 30 min after the stop): HOURS = quota left - 1.3 h
HOURS = 5.0
!python {CODE}/pipeline/hpc/chandra_client.py --queue {ROOT}/chandra_queue_remaining.json --raw {ROOT}/raw --out /kaggle/working/results --workers 16 --hours {HOURS}

In [ ]:
os.killpg(server.pid, 9)
!rm -rf /kaggle/working/code && cd /kaggle/working && zip -q -r results.zip results && ls -la results.zip && wc -l results/chandra_run.jsonl